# Pipeline audit

This notebook audits the persisted outputs of the two-vessel emissions-allocation pipeline against `docs/METHODOLOGY.md`, Sections 0?8. It is deliberately read-only: it neither calls Global Fishing Watch nor rebuilds the pipeline. The executable source of truth remains `src/emissions_allocation/`; this notebook makes the inputs, intermediate artifacts, assertions, and hand-offs visible for review.

Run the pipeline before this notebook. Missing checkpoints are reported as errors rather than recreated here.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import json

import numpy as np
import pandas as pd
from IPython.display import display

from emissions_allocation import activity, allocation, load_config

FIG_SINGLE = (6.5, 4.0)
FIG_MAP = (7.0, 6.0)
FIG_PANEL = (13.0, 4.0)
FIG_GRID = (13.0, 9.0)
DPI_SAVE = 300

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

cfg = load_config()
INTERIM = cfg.path("interim")
OUT = cfg.path("out")

def read_checkpoint(name: str) -> pd.DataFrame:
    """Read one persisted pipeline checkpoint.

    Parameters
    ----------
    name : str
        Filename relative to ``data/interim``.

    Returns
    -------
    pandas.DataFrame
        The checkpoint at its documented grain.

    Raises
    ------
    FileNotFoundError
        If the pipeline has not created the requested artifact.
    """
    checkpoint = INTERIM / name
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint}. Run the pipeline before this audit notebook.")
    return pd.read_parquet(checkpoint)

print(f"Study period: {cfg.start_date} to {cfg.end_date} ({cfg.elapsed_hours:,} elapsed hours)")
print(f"Configured vessels: {[v.imo for v in cfg]}")
print(f"Configured scenarios: {len(cfg.scenarios())}")


Study period: 2017-01-01 to 2024-12-31 (70,128 elapsed hours)
Configured vessels: ['9516454', '9277802']
Configured scenarios: 16


## Pipeline artifacts

The pipeline is staged: raw API responses become hour-, event-, and voyage-level checkpoints; spatial and fuel joins feed scenario-keyed emissions; allocation then joins national baselines. The table below is the audit inventory, not a substitute for the pipeline run.


In [2]:
expected = [
    "vessel_hour_{imo}.parquet", "port_call_{imo}.parquet", "voyage_leg_{imo}.parquet",
    "coverage_{imo}.parquet", "fuel_assignment_{imo}.parquet", "emissions_hour_{imo}.parquet",
    "emissions_year_{imo}.parquet",
]
rows = []
for template in expected:
    for vessel in cfg:
        filename = template.format(imo=vessel.imo)
        artifact = INTERIM / filename
        rows.append({
            "artifact": filename,
            "stage": template.split("_")[0],
            "present": artifact.exists(),
            "bytes": artifact.stat().st_size if artifact.exists() else np.nan,
        })
for filename in ["baseline.parquet", "allocation.parquet", "impacts.parquet", "scenario_spread.parquet"]:
    artifact = INTERIM / filename
    rows.append({"artifact": filename, "stage": "final", "present": artifact.exists(),
                 "bytes": artifact.stat().st_size if artifact.exists() else np.nan})
inventory = pd.DataFrame(rows)
assert inventory.present.all(), "One or more pipeline checkpoints are missing."
display(inventory.style.format({"bytes": "{:,.0f}"}).set_caption("Persisted pipeline artifacts required by this audit"))


,artifact,stage,present,bytes
0,vessel_hour_9516454.parquet,vessel,True,"3,265,723"
1,vessel_hour_9277802.parquet,vessel,True,"3,808,461"
2,port_call_9516454.parquet,port,True,"56,082"
3,port_call_9277802.parquet,port,True,"87,436"
4,voyage_leg_9516454.parquet,voyage,True,"20,575"
5,voyage_leg_9277802.parquet,voyage,True,"31,771"
6,coverage_9516454.parquet,coverage,True,"6,198"
7,coverage_9277802.parquet,coverage,True,"6,175"
8,fuel_assignment_9516454.parquet,fuel,True,"627,664"
9,fuel_assignment_9277802.parquet,fuel,True,"628,937"


## 0. Select vessels

The configuration is the auditable record of the selected hulls. Candidate-discovery outputs are retained separately because discovery is an API operation and must not be repeated merely to read the audit.


In [3]:
selection_rows = []
for vessel in cfg:
    selection_rows.append({
        "IMO": vessel.imo,
        "ship": vessel.shipnames[0],
        "label": vessel.label,
        "flag": vessel.require_spec("flag"),
        "type": vessel.require_spec("ship_type"),
        "DWT (t)": vessel.require_spec("dwt"),
        "names used by presence pull": ", ".join(vessel.shipnames),
    })
display(pd.DataFrame(selection_rows).style.format({"DWT (t)": "{:,.0f}"})
        .set_caption("Section 0: configured pilot vessels"))

for filename in ["vesselB_pool.json", "vesselB_shortlist.json"]:
    source = INTERIM / filename
    if source.exists():
        payload = json.loads(source.read_text(encoding="utf-8"))
        print(f"{filename}: retained discovery payload ({len(payload):,} top-level record(s))")


,IMO,ship,label,flag,type,DWT (t),names used by presence pull
0,9516454,COSCO ITALY,A,HKG,container,"156,610",COSCO ITALY
1,9277802,RCC AMERICA,B,BHS,vehicle,"21,182","RCC AMERICA, HOEGH AMERICA"


vesselB_pool.json: retained discovery payload (2,143 top-level record(s))
vesselB_shortlist.json: retained discovery payload (22 top-level record(s))


## 1. Acquire ship activity data and identify unique ships

The checkpoints below make the three required presence-pull assertions reviewable: non-empty output, exactly one IMO per hull, and coverage measured before any correction. Port calls and voyage legs are retained as independent activity evidence.


In [4]:
activity_rows = []
for vessel in cfg:
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    calls = read_checkpoint(f"port_call_{vessel.imo}.parquet")
    legs = read_checkpoint(f"voyage_leg_{vessel.imo}.parquet")
    assert not hours.empty, f"{vessel.imo}: empty presence checkpoint"
    assert set(hours.imo.astype(str)) == {vessel.imo}, f"{vessel.imo}: identity integrity failure"
    activity_rows.append({
        "IMO": vessel.imo, "ship": vessel.shipnames[0], "vessel-hours": len(hours),
        "distinct IMO": hours.imo.astype(str).nunique(), "port calls": len(calls),
        "port countries": calls.port_iso3.nunique(), "voyage legs": len(legs),
        "international legs": int(legs.is_international.sum()),
    })
display(pd.DataFrame(activity_rows).style.format({"vessel-hours": "{:,}", "port calls": "{:,}",
                                                  "voyage legs": "{:,}", "international legs": "{:,}"})
        .set_caption("Section 1: identity and activity checkpoint assertions"))


,IMO,ship,vessel-hours,distinct IMO,port calls,port countries,voyage legs,international legs
0,9516454,COSCO ITALY,"70,128",1,389,17,388,225
1,9277802,RCC AMERICA,"70,128",1,588,64,587,395


In [5]:
coverage = pd.concat([
    read_checkpoint(f"coverage_{vessel.imo}.parquet").assign(ship=vessel.shipnames[0])
    for vessel in cfg
], ignore_index=True)
assert coverage.coverage_active.between(0, 1).all()
display(coverage[["ship", "imo", "year", "elapsed_hours", "inactive_hours", "observed_hours",
                  "coverage_raw", "coverage_active"]]
        .style.format({"elapsed_hours": "{:,}", "inactive_hours": "{:,}", "observed_hours": "{:,}",
                       "coverage_raw": "{:.2%}", "coverage_active": "{:.2%}"})
        .set_caption("Section 1.7: observed coverage before emissions correction"))

bias_rows = []
for vessel in cfg:
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    active = hours.loc[~hours.is_inactive]
    for window in cfg.run["smoothing_windows"]:
        bias_rows.append({"IMO": vessel.imo, "ship": vessel.shipnames[0], "window (h)": window,
                          "mean(v?) / mean(v)?": activity.cubic_bias(active[f"sog_w{window}"])})
display(pd.DataFrame(bias_rows).style.format({"mean(v?) / mean(v)?": "{:.2f}?"})
        .set_caption("Section 1.6: cubic-speed bias by smoothing window"))


,ship,imo,year,elapsed_hours,inactive_hours,observed_hours,coverage_raw,coverage_active
0,COSCO ITALY,9516454,2017,"8,760",0,"7,185",82.02%,82.02%
1,COSCO ITALY,9516454,2018,"8,760","1,708","6,233",71.15%,88.39%
2,COSCO ITALY,9516454,2019,"8,760","4,906","3,159",36.06%,81.97%
3,COSCO ITALY,9516454,2020,"8,784","1,866","6,388",72.72%,92.34%
4,COSCO ITALY,9516454,2021,"8,760",0,"8,163",93.18%,93.18%
5,COSCO ITALY,9516454,2022,"8,760",0,"8,498",97.01%,97.01%
6,COSCO ITALY,9516454,2023,"8,760",0,"8,743",99.81%,99.81%
7,COSCO ITALY,9516454,2024,"8,784",0,"8,782",99.98%,99.98%
8,RCC AMERICA,9277802,2017,"8,760",0,"7,501",85.63%,85.63%
9,RCC AMERICA,9277802,2018,"8,760",0,"7,771",88.71%,88.71%


,IMO,ship,window (h),mean(v?) / mean(v)?
0,9516454,COSCO ITALY,1,1.94?
1,9516454,COSCO ITALY,3,1.38?
2,9516454,COSCO ITALY,5,1.38?
3,9516454,COSCO ITALY,7,1.41?
4,9277802,RCC AMERICA,1,2.06?
5,9277802,RCC AMERICA,3,1.49?
6,9277802,RCC AMERICA,5,1.49?
7,9277802,RCC AMERICA,7,1.52?


## 2. Acquire registry data and complete ship specifications

Every derived parameter is stored with its value, source, method, and estimated flag. This is the substitution point for a researcher who later obtains a sourced installed power or service speed.


In [6]:
spec_rows = []
for vessel in cfg:
    for name, parameter in vessel.specs.items():
        spec_rows.append({
            "IMO": vessel.imo, "parameter": name, "value": parameter.value, "unit": parameter.unit,
            "estimated": parameter.estimated, "source": parameter.source, "method": parameter.method,
        })
spec_table = pd.DataFrame(spec_rows)
assert spec_table.loc[spec_table.estimated & spec_table.value.notna(), ["source", "method"]].notna().all().all()
display(spec_table.style.set_caption("Section 2: configured vessel specifications and provenance"))


,IMO,parameter,value,unit,estimated,source,method
0,9516454,mmsi,477845600,nan,False,GFW presence + Equasis,observed
1,9516454,callsign,VRNE4,nan,False,GFW registryInfo + Equasis,observed
2,9516454,flag,HKG,nan,False,Equasis; GFW registryInfo.flag,observed
3,9516454,ship_type,container,nan,False,Equasis,observed
4,9516454,year_built,2014,nan,False,Equasis,observed
5,9516454,dwt,156610,t,False,Equasis,observed
6,9516454,gt,154592,t,False,Equasis (since 2023),observed
7,9516454,gt_gfw,153666,t,False,GFW registryInfo.tonnageGt,observed
8,9516454,loa_m,365.900000,nan,False,public vessel registers,observed
9,9516454,beam_m,51.200000,nan,False,public vessel registers,observed


## 3. Assign fuel type and emission factors

This checkpoint is one row per vessel-hour before scenario expansion. It makes the ECA and EU-to-EU-leg switches inspectable, and confirms that every hour receives exactly one fuel type.


In [7]:
fuel_rows = []
for vessel in cfg:
    fuel = read_checkpoint(f"fuel_assignment_{vessel.imo}.parquet")
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    assert len(fuel) == len(hours), f"{vessel.imo}: fuel assignment does not cover every vessel-hour"
    assert fuel.fuel_type.notna().all(), f"{vessel.imo}: unassigned fuel type"
    fuel_rows.append({
        "IMO": vessel.imo, "ship": vessel.shipnames[0], "hours": len(fuel),
        "ECA hours": int(fuel.in_eca.sum()), "EU?EU-leg hours": int(fuel.is_eu_eu_leg.sum()),
        "MDO/MGO hours": int((fuel.fuel_type == "mdo").sum()), "HFO hours": int((fuel.fuel_type == "hfo").sum()),
    })
display(pd.DataFrame(fuel_rows).style.format({c: "{:,}" for c in ["hours", "ECA hours", "EU?EU-leg hours", "MDO/MGO hours", "HFO hours"]})
        .set_caption("Section 3: fuel-assignment coverage and triggers"))


,IMO,ship,hours,ECA hours,EU?EU-leg hours,MDO/MGO hours,HFO hours
0,9516454,COSCO ITALY,"70,128","4,310","1,532",0,0
1,9277802,RCC AMERICA,"70,128","8,489","3,084",0,0


## 4. Calculate CO2 emissions

The hour-level output is expanded by every valid power-estimate and smoothing-window scenario. The sample shows the full hand-off from operating mode and load through fuel consumption to CO?; the annual table checks that the aggregation is retained.


In [8]:
hourly_rows = []
for vessel in cfg:
    hourly = read_checkpoint(f"emissions_hour_{vessel.imo}.parquet")
    expected_scenarios = len(vessel.resolve_power_estimates(cfg.run["power_estimates"])) * len(cfg.run["smoothing_windows"])
    assert hourly.scenario_id.nunique() == expected_scenarios, f"{vessel.imo}: incomplete scenario expansion"
    assert (hourly.fc_total_g >= 0).all() and (hourly.co2_tonnes >= 0).all()
    hourly_rows.append({"IMO": vessel.imo, "ship": vessel.shipnames[0], "hour-scenario rows": len(hourly),
                        "scenarios": hourly.scenario_id.nunique(), "operating modes": hourly.operating_mode.nunique()})
display(pd.DataFrame(hourly_rows).style.format({"hour-scenario rows": "{:,}"})
        .set_caption("Section 4: scenario-keyed hourly emissions assertions"))

sample = read_checkpoint(f"emissions_hour_{cfg.vessels[0].imo}.parquet")
sample = sample[(sample.power_estimate == "A") & (sample.smoothing_window == 3)].head(8)
display(sample[["ts", "operating_mode", "sog", "me_load", "fuel_type", "w_me_kw", "w_ae_kw", "w_bo_kw",
                "fc_total_g", "co2_tonnes"]].style.format({"sog": "{:.2f}", "me_load": "{:.3f}",
                                                           "w_me_kw": "{:,.0f}", "w_ae_kw": "{:,.0f}",
                                                           "w_bo_kw": "{:,.0f}", "fc_total_g": "{:,.0f}",
                                                           "co2_tonnes": "{:.3f}"})
        .set_caption("Section 4: representative hourly emissions records (vessel A, estimate A, w=3)"))


,IMO,ship,hour-scenario rows,scenarios,operating modes
0,9516454,COSCO ITALY,"493,184",8,5
1,9277802,RCC AMERICA,"274,308",4,5


,ts,operating_mode,sog,me_load,fuel_type,w_me_kw,w_ae_kw,w_bo_kw,fc_total_g,co2_tonnes
61648,2017-01-01 00:00:00,slow_transit,16.96,0.292,HFO,"19,858","2,050",0,"4,261,585",13.271
61649,2017-01-01 01:00:00,slow_transit,16.96,0.292,HFO,"19,858","2,050",0,"4,261,585",13.271
61650,2017-01-01 02:00:00,slow_transit,14.43,0.180,HFO,"12,225","2,050",0,"2,896,288",9.019
61651,2017-01-01 03:00:00,slow_transit,15.09,0.206,HFO,"13,986","2,050",0,"3,221,901",10.033
61652,2017-01-01 04:00:00,slow_transit,14.75,0.192,HFO,"13,054","2,050",0,"3,050,408",9.499
61653,2017-01-01 05:00:00,slow_transit,17.76,0.336,HFO,"22,805","2,050",0,"4,761,410",14.827
61654,2017-01-01 06:00:00,slow_transit,18.41,0.374,HFO,"25,382","2,050",0,"5,189,010",16.159
61655,2017-01-01 07:00:00,slow_transit,16.21,0.255,HFO,"17,349","2,050",0,"3,825,413",11.912


In [9]:
annual = read_checkpoint("emissions_year.parquet")
assert annual.co2_tonnes.notna().all() and (annual.co2_tonnes >= 0).all()
annual_audit = annual[(annual.power_estimate == "A") & (annual.smoothing_window == 3)]
display(annual_audit[["imo", "year", "modelled_hours", "coverage_active", "co2_tonnes_observed",
                      "co2_tonnes_corrected", "co2_tonnes", "is_low_confidence"]]
        .style.format({"modelled_hours": "{:,}", "coverage_active": "{:.2%}",
                       "co2_tonnes_observed": "{:,.0f}", "co2_tonnes_corrected": "{:,.0f}",
                       "co2_tonnes": "{:,.0f}"})
        .set_caption("Section 4.5: annual CO? aggregation (estimate A, w=3)"))


,imo,year,modelled_hours,coverage_active,co2_tonnes_observed,co2_tonnes_corrected,co2_tonnes,is_low_confidence
1,9516454,2017,"8,760",82.02%,"90,033","109,768","109,768",True
9,9516454,2018,"7,052",88.39%,"65,704","74,338","74,338",True
17,9516454,2019,"3,854",81.97%,"28,120","34,307","34,307",True
25,9516454,2020,"6,918",92.34%,"71,446","77,374","77,374",True
33,9516454,2021,"8,760",93.18%,"93,890","100,757","100,757",True
41,9516454,2022,"8,760",97.01%,"87,887","90,597","90,597",False
49,9516454,2023,"8,760",99.81%,"72,200","72,340","72,340",False
57,9516454,2024,"8,784",99.98%,"83,183","83,202","83,202",False
65,9277802,2017,"8,760",85.63%,"31,408","36,679","36,679",True
69,9277802,2018,"8,760",88.71%,"26,868","30,288","30,288",True


## 5. Classify international ships and allocate emissions

The domestic test is a persisted SQL output. Only international hulls are in scope. The allocation table then makes each role-based country assignment explicit; bunker-fuel allocation remains intentionally absent because individual refuelling locations are unavailable.


In [10]:
domestic = pd.read_csv(OUT / "domestic_test.csv")
assert domestic.is_international.all(), "A configured pilot vessel is classified as domestic."
display(domestic.style.format({"dominant_eez_hours": "{:,}", "hours_in_any_eez": "{:,}",
                               "hours_disputed": "{:,}", "dominant_eez_share": "{:.2%}"})
        .set_caption("Section 5.4: EEZ-based international/domestic classification"))

for treatment in cfg.run["hk_treatments"]:
    keys = allocation.summarise_options(cfg, treatment)
    display(keys.style.set_caption(f"Section 5.1: allocation keys, Hong Kong treatment: {treatment}"))


,imo,dominant_eez_iso3,dominant_eez_hours,hours_in_any_eez,hours_disputed,dominant_eez_share,is_domestic,is_international
0,9277802,JPN,"3,609","50,619.0",111.0,7.13%,False,True
1,9516454,CHN,"11,825","39,481.0",533.0,29.95%,False,True


option,imo,flag,manager,operator,owner,n_distinct_countries,is_degenerate,hk_treatment
0,9277802,BHS,GRC,IMN,IMN,3,False,separate
1,9516454,HKG,CHN,CHN,CHN,2,False,separate


option,imo,flag,manager,operator,owner,n_distinct_countries,is_degenerate,hk_treatment
0,9277802,BHS,GRC,IMN,IMN,3,False,folded_into_china
1,9516454,HKG,CHN,CHN,CHN,1,True,folded_into_china


## 6?7. Establish baselines and compute allocation impacts

The baseline is in MtCO? and is joined to allocation results by territory treatment. This audit exposes both the unit conversion result and the final impact records without interpreting their country rankings at n = 2.


In [11]:
baseline = read_checkpoint("baseline.parquet")
allocation_result = read_checkpoint("allocation.parquet")
impacts = read_checkpoint("impacts.parquet")
assert baseline.mtco2.notna().all()
assert impacts.baseline_mt.notna().all()
assert impacts.delta_e_pct.notna().all()

base_2024 = baseline[(baseline.year == 2024) & baseline.country.isin(["China", "Hong Kong"])]
display(base_2024.style.format({"mtc": "{:,.2f}", "mtco2": "{:,.2f}"})
        .set_caption("Section 6: 2024 GCB baselines used by the Hong Kong sensitivity"))

impact_2024 = impacts[(impacts.year == 2024) & (impacts.power_estimate == "A") & (impacts.smoothing_window == 3)]
display(impact_2024[["option", "country", "gcb_name", "hk_treatment", "delta_e_mt", "baseline_mt", "delta_e_pct"]]
        .sort_values(["option", "hk_treatment", "country"])
        .style.format({"delta_e_mt": "{:.5f}", "baseline_mt": "{:,.1f}", "delta_e_pct": "{:.6f}"})
        .set_caption("Section 7: 2024 allocation impacts (estimate A, w=3)"))


,year,country,mtc,mtco2,hk_treatment
311,2024,China,"3,353.99","12,289.04",separate
671,2024,Hong Kong,9.09,33.32,separate
2199,2024,China,"3,363.09","12,322.36",folded_into_china


,option,country,gcb_name,hk_treatment,delta_e_mt,baseline_mt,delta_e_pct
171,flag,BHS,Bahamas,folded_into_china,0.02032,3.1,0.662009
170,flag,HKG,China,folded_into_china,0.08320,"12,322.4",0.000675
183,flag,BHS,Bahamas,separate,0.02032,3.1,0.662009
182,flag,HKG,Hong Kong,separate,0.08320,33.3,0.249678
362,manager,CHN,China,folded_into_china,0.08320,"12,322.4",0.000675
363,manager,GRC,Greece,folded_into_china,0.02032,53.4,0.038083
374,manager,CHN,China,separate,0.08320,"12,289.0",0.000677
375,manager,GRC,Greece,separate,0.02032,53.4,0.038083
554,operator,CHN,China,folded_into_china,0.08320,"12,322.4",0.000675
555,operator,IMN,United Kingdom,folded_into_china,0.02032,312.9,0.006494


## 8. Sensitivity and validation

The final audit stage confirms that the configured scenario space reaches the impact outputs and displays the validation records written by the pipeline. A warning or failure remains evidence, not something this notebook suppresses.


In [12]:
spread = read_checkpoint("scenario_spread.parquet")
valid_scenario_counts = {len(vessel.resolve_power_estimates(cfg.run["power_estimates"])) * len(cfg.run["smoothing_windows"]) for vessel in cfg}
assert spread.n_scenarios.isin(valid_scenario_counts).all(), "Scenario spread contains an unexpected scenario count."
display(spread[spread.year == 2024][["option", "country", "hk_treatment", "delta_e_mt_min", "delta_e_mt_max",
                                     "spread_ratio", "n_scenarios"]]
        .style.format({"delta_e_mt_min": "{:.5f}", "delta_e_mt_max": "{:.5f}", "spread_ratio": "{:.2f}x"})
        .set_caption("Section 8.1: scenario spread in 2024"))

validation = pd.concat([
    pd.read_csv(OUT / f"validation_{vessel.imo}.csv").assign(IMO=vessel.imo, ship=vessel.shipnames[0])
    for vessel in cfg
], ignore_index=True)
display(validation[["ship", "IMO", "check", "status", "detail", "basis"]]
        .style.set_caption("Section 8.2: persisted validation results"))
print(validation.groupby("status").size().rename("checks").to_string())


,option,country,hk_treatment,delta_e_mt_min,delta_e_mt_max,spread_ratio,n_scenarios
7,flag,BHS,folded_into_china,0.01947,0.02177,1.12x,4
15,flag,BHS,separate,0.01947,0.02177,1.12x,4
23,flag,HKG,folded_into_china,0.07964,0.13301,1.67x,8
31,flag,HKG,separate,0.07964,0.13301,1.67x,8
39,manager,CHN,folded_into_china,0.07964,0.13301,1.67x,8
47,manager,CHN,separate,0.07964,0.13301,1.67x,8
55,manager,GRC,folded_into_china,0.01947,0.02177,1.12x,4
63,manager,GRC,separate,0.01947,0.02177,1.12x,4
71,operator,CHN,folded_into_china,0.07964,0.13301,1.67x,8
79,operator,CHN,separate,0.07964,0.13301,1.67x,8


,ship,IMO,check,status,detail,basis
0,COSCO ITALY,9516454,Identity integrity,PASS,one IMO throughout: 9516454,distinct IMO in vessel_hour
1,COSCO ITALY,9516454,Hour conservation,WARN,"8 years, active coverage 82.0%-100.0%; below 95% in 5 year(s): 2017 82.0%, 2018 88.4%, 2019 82.0%, 2020 92.3%, 2021 93.2%",observed hours / in-service hours
2,COSCO ITALY,9516454,Leg-speed plausibility,PASS,"346 legs with movement, median 13.2 kn, 95th pct 19.3 kn; 5 legs above 30.0 kn are anchorage-segmentation artefacts, not voyages (5/5 same-country, 0 same-port, 1.0 h = 0.00% of leg time)",great-circle distance / leg duration; impossible legs diagnosed
3,COSCO ITALY,9516454,Port-call/track agreement,PASS,"20,107 h at berth/anchored/manoeuvring against 17,427 h inside port visits (ratio 1.15)",modes vs event intervals
4,COSCO ITALY,9516454,Fleet envelope,FAIL,A at 25.55 kn outside the published range (inside: B),observed service-speed range for the ship type
5,COSCO ITALY,9516454,Smoothing sensitivity,PASS,w=1:1.94x w=3:1.38x w=5:1.38x w=7:1.41x (lowest at w=3),mean(v^3)/(mean v)^3 over in-service hours
6,COSCO ITALY,9516454,THETIS-MRV,PASS,"1 year(s) comparable in MRV scope; modelled/verified A=0.64x, B=0.98x (closest: B); at_dock covers 17% of calls",modelled emissions restricted to MRV scope against EMSA-verified figures -- the only external ground truth in this project
7,RCC AMERICA,9277802,Identity integrity,PASS,one IMO throughout: 9277802,distinct IMO in vessel_hour
8,RCC AMERICA,9277802,Hour conservation,WARN,"8 years, active coverage 84.5%-99.1%; below 95% in 5 year(s): 2017 85.6%, 2018 88.7%, 2019 87.1%, 2020 88.4%, 2021 84.5%",observed hours / in-service hours
9,RCC AMERICA,9277802,Leg-speed plausibility,PASS,"515 legs with movement, median 11.5 kn, 95th pct 17.2 kn; 1 legs above 30.0 kn are anchorage-segmentation artefacts, not voyages (1/1 same-country, 0 same-port, 0.1 h = 0.00% of leg time)",great-circle distance / leg duration; impossible legs diagnosed


status
FAIL       1
PASS       8
PENDING    1
WARN       4


## Audit boundary

This notebook demonstrates every persisted hand-off in the current pipeline. It intentionally does not expose credentials, repeat live API calls, or replace the implementation modules. Open methodological items remain visible through the pipeline's validation output and configuration rather than being silently filled in.
